# Siniestralidad vial — Ciudad de Chicago
## Fase 3: Análisis exploratorio y diagnóstico

**Cliente:** Oficina de Seguridad Vial, Departamento de Transporte de la Ciudad de Chicago (CDOT) — Observatorio Vial
**Proveedor:** Grupo 1 de Ciencia de Datos
Objetivos de esta fase (según la propuesta):

1. Analizar la **composición** de los siniestros: severidad, tipos de choque y causas.
2. Analizar **patrones temporales** (hora, día, mes) cruzados con **clima e iluminación**.
3. Construir un análisis de **puntos críticos** cruzando ubicación con severidad.
4. **Segmentar por causa principal** para orientar la inversión (fiscalización, infraestructura o concientización).
5. **Documentar las limitaciones** de calidad del dato que puedan afectar los hallazgos.

### Definiciones de severidad usadas en todo el notebook

| Indicador | Definición | Uso |
|---|---|---|
| **Con heridos** | `Lesiones totales > 0` (cualquier lesión, incluida la declarada no evidente) | Línea base pedida por el cliente |
| **Grave (KSI)** | Al menos una lesión **fatal o incapacitante** | Métrica principal de riesgo (*Killed or Seriously Injured*, estándar internacional) |
| **Fatal** | Al menos una lesión fatal | Se informa, pero es muy poco frecuente para segmentar |

Se usa la **tasa de graves** (% de siniestros que resultan graves) como medida de *riesgo real*, y el **volumen** de siniestros como medida de *exposición*. Un factor puede tener mucho volumen y baja severidad (y viceversa): por eso casi todos los gráficos muestran ambas dimensiones lado a lado.

## 0. Librerías y configuración

In [ ]:
import textwrap
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from config import ANIO_DESDE, ANIO_HASTA
from preparacion import (
    COLS_GEO,
    DIAS,
    FRANJAS,
    MAPA_COLUMNAS,
    MESES,
    NIVELES,
    ORDEN_CLIMA,
    ORDEN_LUZ,
    PALANCA,
    cargar_base,
    preparar_base,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

# Paleta: un azul para volumen, rojo solo para resaltar severidad, gris para referencias.
AZUL, AZUL_CLARO, ROJO, NARANJA, VERDE = "#2a78d6", "#86b6ef", "#e34948", "#eb6834", "#1baf7a"
GRIS, TINTA, TINTA2, GRILLA, FONDO = "#898781", "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
CMAP_AZUL = sns.blend_palette(["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"], as_cmap=True)
CMAP_ROJO = sns.blend_palette(["#fdf3f2", "#f2a3a2", "#e34948", "#8f1d1c"], as_cmap=True)

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA2,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "text.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
})

### Funciones auxiliares

- `wilson_ic`: intervalo de confianza del 95 % para una proporción. Es clave porque muchas categorías tienen pocos casos y una tasa "alta" puede ser solo ruido.
- `tabla_tasas`: para una variable categórica devuelve volumen, % del total, tasa de heridos, tasa de graves con IC y **riesgo relativo (RR)** respecto del promedio general.
- `panel_volumen_severidad`: gráfico de dos paneles con el mismo orden de categorías — volumen a la izquierda y tasa de graves (con IC) a la derecha.
- `cramers_v`: fuerza de asociación entre dos variables categóricas (0 = nula, 1 = total).

In [ ]:
def wilson_ic(k, n, z=1.96):
    # Intervalo de Wilson para una proporción k/n.
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.where(n > 0, k / n, np.nan)
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return centro - margen, centro + margen


def tabla_tasas(data, col, min_n=0, orden=None):
    t = data.groupby(col, observed=True).agg(
        siniestros=("grave", "size"),
        con_heridos=("con_heridos", "sum"),
        graves=("grave", "sum"),
        fatales=("fatal", "sum"),
    )
    t["pct_total"] = t["siniestros"] / t["siniestros"].sum() * 100
    t["tasa_heridos"] = t["con_heridos"] / t["siniestros"] * 100
    t["tasa_graves"] = t["graves"] / t["siniestros"] * 100
    lo, hi = wilson_ic(t["graves"], t["siniestros"])
    t["ic_inf"], t["ic_sup"] = lo * 100, hi * 100
    t["RR_graves"] = t["tasa_graves"] / (data["grave"].mean() * 100)
    t = t[t["siniestros"] >= min_n]
    t = t.reindex([o for o in orden if o in t.index]) if orden is not None else t.sort_values("siniestros", ascending=False)
    return t[["siniestros", "pct_total", "con_heridos", "tasa_heridos", "graves", "tasa_graves",
              "ic_inf", "ic_sup", "RR_graves", "fatales"]]


def panel_volumen_severidad(data, col, titulo, min_n=100, top=None, orden=None, figsize=None):
    t = tabla_tasas(data, col, min_n=min_n, orden=orden)
    if top:
        t = t.head(top)
    t = t.iloc[::-1]  # la categoría más frecuente arriba
    etiquetas = [str(i).capitalize() if len(str(i)) <= 55 else str(i)[:52].capitalize() + "…" for i in t.index]
    y = np.arange(len(t))
    fig, (a1, a2) = plt.subplots(1, 2, sharey=True, figsize=figsize or (12, 0.38 * len(t) + 1.4),
                                 gridspec_kw={"width_ratios": [1.1, 1]})
    a1.barh(y, t["siniestros"], color=AZUL, height=0.7)
    for yi, (n, p) in enumerate(zip(t["siniestros"], t["pct_total"])):
        a1.text(n, yi, f"  {n:,.0f} ({p:.1f}%)", va="center", fontsize=8, color=TINTA2)
    a1.set_yticks(y, etiquetas)
    a1.set_xlim(0, t["siniestros"].max() * 1.35)
    a1.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    a1.set_title("Volumen de siniestros")
    a1.grid(axis="y", visible=False)

    tasa_global = data["grave"].mean() * 100
    colores = [ROJO if lo > tasa_global else (AZUL if hi < tasa_global else GRIS)
               for lo, hi in zip(t["ic_inf"], t["ic_sup"])]
    a2.hlines(y, t["ic_inf"], t["ic_sup"], color=colores, lw=2)
    a2.scatter(t["tasa_graves"], y, color=colores, s=45, zorder=3)
    a2.axvline(tasa_global, color=TINTA2, ls="--", lw=1)
    a2.text(tasa_global, len(t) - 0.4, f" promedio {tasa_global:.1f}%", fontsize=8, color=TINTA2)
    a2.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    a2.set_title("% de siniestros graves (IC 95%)")
    a2.grid(axis="y", visible=False)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.text(0.01, -0.01, "Rojo: significativamente por encima del promedio · Azul: por debajo · Gris: no distinguible. "
             f"Se excluyen categorías con menos de {min_n} casos.", fontsize=8, color=TINTA2)
    fig.tight_layout()
    plt.show()
    return t.iloc[::-1]


def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    chi2, p, dof, _ = stats.chi2_contingency(tab)
    n = tab.to_numpy().sum()
    r, k = tab.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))), chi2, p


def heatmap(tabla, titulo, cmap, fmt=".0f", ax=None, cbar_label="", mask=None, **kw):
    ax = ax or plt.gca()
    sns.heatmap(tabla, cmap=cmap, annot=True, fmt=fmt, linewidths=1, linecolor=FONDO, mask=mask,
                cbar_kws={"label": cbar_label, "shrink": 0.8}, ax=ax, annot_kws={"fontsize": 7}, **kw)
    ax.set_title(titulo)
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels([textwrap.fill(t.get_text().capitalize(), 16) for t in ax.get_xticklabels()], rotation=0, fontsize=8)
    ax.set_yticklabels([t.get_text().capitalize() for t in ax.get_yticklabels()], rotation=0)
    return ax

## 1. Carga y preparación para el análisis

El dataset viene de la Fase 2 (auditoría y limpieza). La preparación **analítica** (nombres de columnas cortos, tipos de dato correctos y variables derivadas) está en [`preparacion.py`](preparacion.py): este notebook, la Fase 4 y el tablero de la Fase 5 la consumen de ahí, así que los números de las tres piezas salen de la misma base. Acá solo se carga y se documenta el contrato.

**Coordenadas:** la base del Observatorio no trae ubicación. El script [`enriquecer_coordenadas.py`](enriquecer_coordenadas.py) las recupera del portal de datos abiertos de la Ciudad de Chicago (*Traffic Crashes – Crashes*) vinculando cada registro con el siniestro original, y genera `dataset_geo.parquet` (la misma base + latitud, longitud y dirección). El detalle del vínculo y su calidad está en la sección 2.6.

> Si `dataset_geo.parquet` no existe, hay que correr antes `python enriquecer_coordenadas.py` (descarga ~1 millón de registros del portal; tarda unos minutos solo la primera vez).

In [ ]:
df_full = preparar_base()
print(f"Registros: {len(df_full):,} · Columnas: {df_full.shape[1]} · Años: {df_full['anio'].min()}-{df_full['anio'].max()}")
df_full.head(3)


In [ ]:
# El renombrado vive en preparacion.py y es por nombre, nunca por posición: si la base no trae
# exactamente estas columnas, la carga falla. Acá se muestra el contrato vigente.
diccionario = pd.DataFrame(MAPA_COLUMNAS.items(), columns=["columna_original", "nombre_analisis"])
diccionario


In [ ]:
# preparacion.py ya convirtió los tipos, validó que el mes y el día de la semana coincidan con la
# fecha y que ninguna categoría de clima, iluminación, severidad o causa quede sin clasificar.
# Acá solo se muestran las variables derivadas que usan las secciones siguientes.
df_full[["fecha", "dia", "mes_nombre", "franja", "clima_g", "luz_g", "severidad", "familia_causa", "palanca"]].head()


### Duplicados y período de análisis

- **Duplicados:** la base no tiene un identificador de siniestro, así que dos filas idénticas en *todos* los campos (misma fecha, hora, entorno, causa y lesiones) se consideran carga duplicada y se eliminan.
- **Período:** el brief pide analizar los últimos 3 a 5 años. Como se ve en la sección 2, los años 2013–2017 tienen cobertura incompleta (la carga se fue incorporando de a poco) y 2025 solo tiene unas semanas. Para que las comparaciones entre años sean justas, **el análisis principal usa los años completos 2018–2024** (7 años, incluye los últimos 5).

In [ ]:
df = cargar_base()
print(f"Duplicados exactos eliminados: {df_full.attrs['duplicados_eliminados']}")
print(f"Base de análisis {ANIO_DESDE}-{ANIO_HASTA}: {len(df):,} siniestros "
      f"({len(df) / len(df_full):.1%} del total; se excluyen {len(df_full) - len(df):,} registros de años incompletos)")


## 2. Diagnóstico de calidad del dato

Antes de interpretar patrones, conviene saber **qué tan confiable es cada campo**. El cliente ya advirtió que la carga policial tiene baches y causas subjetivas; acá se cuantifica.

### 2.1 Campos solicitados en el brief vs. campos disponibles

In [ ]:
pedido = [
    ("Fecha, hora, día de la semana, mes", "Sí", "fecha, hora, dia_semana, mes"),
    ("Fecha de notificación a la policía", "No", "—"),
    ("Clima, iluminación, superficie, defectos, alineación", "Sí", "clima, iluminacion, superficie, defecto_via, alineacion"),
    ("Latitud y longitud", "Recuperado", "latitud, longitud, calle (vinculadas desde el portal de datos abiertos, ver 2.6)"),
    ("Tipo de vía", "Sí", "tipo_via (mezcla tipo de calzada y tipo de intersección)"),
    ("Cantidad de carriles", "No", "—"),
    ("Ocurrió en intersección", "Sí", "en_interseccion"),
    ("Dispositivo de control de tránsito", "Sí (extra)", "dispositivo_control"),
    ("Tipo de choque", "Sí", "primer_tipo_choque"),
    ("Fuga del responsable", "No", "—"),
    ("Cantidad de vehículos, magnitud del daño", "Sí", "vehiculos, danos"),
    ("Causa principal", "Sí", "causa_principal"),
    ("Causa secundaria", "No", "—"),
    ("Lesión más grave y desglose por gravedad", "Sí", "lesion_mas_grave, lesiones_*"),
]
pd.DataFrame(pedido, columns=["Información descripta en el brief", "¿Disponible?", "Campo"])

### 2.2 Valores no informativos por campo

No hay celdas vacías (`NaN`), pero el faltante está "disfrazado" de categorías como *DESCONOCIDO*, *NO SE PUDO DETERMINAR* u *OTRO*.

In [ ]:
NO_INFORMATIVOS = {"DESCONOCIDO", "NO SE PUDO DETERMINAR", "NO APLICA", "NO REPORTADO",
                   "TIPO DE INTERSECCIÓN DESCONOCIDO", "OTRO", "OTRO OBJETO"}
cols_cat = ["causa_principal", "defecto_via", "superficie", "clima", "dispositivo_control",
            "iluminacion", "tipo_via", "primer_tipo_choque", "alineacion"]
print("Celdas vacías (NaN) en la base sin columnas de ubicación:", int(df_full.drop(columns=list(COLS_GEO)).isna().sum().sum()))

calidad = pd.DataFrame({
    c: {
        "Desconocido / indeterminado": df[c].isin(NO_INFORMATIVOS - {"OTRO", "OTRO OBJETO"}).mean() * 100,
        "Otro (sin especificar)": df[c].isin({"OTRO", "OTRO OBJETO"}).mean() * 100,
    } for c in cols_cat
}).T.sort_values("Desconocido / indeterminado")

ax = calidad.plot.barh(stacked=True, color=[ROJO, AZUL_CLARO], figsize=(9, 4), width=0.7, edgecolor=FONDO)
for i, total in enumerate(calidad.sum(axis=1)):
    ax.text(total, i, f"  {total:.1f}%", va="center", fontsize=8, color=TINTA2)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("% de registros sin información útil, por campo")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
calidad.round(1)

### 2.3 Cobertura temporal

In [ ]:
por_anio = df_full.groupby("anio").size()
fig, ax = plt.subplots(figsize=(10, 3.2))
colores = [AZUL if ANIO_DESDE <= a <= ANIO_HASTA else GRIS for a in por_anio.index]
ax.bar(por_anio.index.astype(str), por_anio.values, color=colores, width=0.7)
for x, v in enumerate(por_anio.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=8, color=TINTA2)
ax.set_title("Registros por año (gris: años incompletos, excluidos del análisis principal)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

### 2.4 Precisión de la hora: redondeo en la carga

Si la hora se registrara con precisión, los minutos deberían distribuirse de forma pareja (~1,7 % cada uno). Un exceso en :00, :30, :15, :45 indica que el oficial **estima** la hora. Esto no invalida el análisis por hora, pero sí desaconseja análisis más finos que la hora.

In [ ]:
minutos = df["minuto"].value_counts(normalize=True).sort_index() * 100
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(minutos.index, minutos.values, color=[ROJO if m % 15 == 0 else AZUL_CLARO for m in minutos.index], width=0.8)
ax.axhline(100 / 60, color=TINTA2, ls="--", lw=1)
ax.text(60, 100 / 60, " esperado si\n fuera exacto", fontsize=8, color=TINTA2, va="center")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("Minuto registrado"); ax.set_title("Distribución del minuto de la hora del siniestro")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
print(f"Registros en minutos múltiplos de 15: {df['minuto'].mod(15).eq(0).mean():.1%} (esperado: 6,7%)")

### 2.5 Representatividad y consistencia interna

In [ ]:
print("En intersección:")
print(df["en_interseccion"].value_counts(normalize=True).mul(100).round(1).to_string(), "\n")

# ¿'lesion_mas_grave' coincide con el desglose numérico de lesiones?
esperado = np.select(
    [df["lesiones_fatales"] > 0, df["lesiones_incapacitantes"] > 0, df["lesiones_no_incapacitantes"] > 0,
     df["lesiones_no_evidentes"] > 0], NIVELES[:4], default="Sin lesión")
inconsistentes = (df["severidad"].astype(str) != esperado).sum()
print(f"Registros donde la 'lesión más grave' no coincide con el desglose numérico: {inconsistentes:,} "
      f"({inconsistentes / len(df):.2%})\n")

# El campo 'tipo_choque' mezcla heridos con 'necesitó grúa'
print(pd.crosstab(df["severidad"], df["tipo_choque"], margins=True))

### 2.6 Calidad de la georreferenciación

Cada registro se vinculó con el portal de datos abiertos por **fecha y hora exactas + cantidad de vehículos + desglose de lesiones + causa principal**. Si había más de un candidato, se desempató por tipo de choque, clima, iluminación, tipo de vía, dispositivo de control y superficie.

- **único:** un solo siniestro del portal coincide en todos los campos clave.
- **desempate:** varios candidatos, resuelto por los campos de desempate.
- **sin vínculo:** no se encontró coincidencia o el empate no se pudo resolver.
- **vinculado sin coordenadas:** el siniestro existe en el portal pero sin ubicación (o con una ubicación fuera de la ciudad).

Para que el mapa no esté sesgado, también se verifica que los siniestros con y sin coordenadas tengan una severidad parecida.

In [ ]:
geo_calidad = df.groupby("calidad_geo").agg(siniestros=("grave", "size"), tasa_graves=("grave", "mean"))
geo_calidad["pct"] = geo_calidad["siniestros"] / len(df) * 100
geo_calidad["tasa_graves"] *= 100
display(geo_calidad.sort_values("siniestros", ascending=False).round(2))

con_coord = df["latitud"].notna()
print(f"Siniestros con coordenadas: {con_coord.mean():.1%} ({con_coord.sum():,})")
print(f"% graves con coordenadas: {df.loc[con_coord, 'grave'].mean():.2%} · sin coordenadas: {df.loc[~con_coord, 'grave'].mean():.2%}")

**Lectura de la sección 2**

- La **causa principal** es el campo más débil: alrededor de 3 de cada 10 siniestros no tienen causa determinada. Todo análisis por causa describe solo a los siniestros *con* causa asignada, y esa causa es la **opinión del oficial** en el momento.
- **Defecto de la vía** y **superficie** también tienen mucho desconocido, así que los hallazgos sobre infraestructura deben tomarse con cautela.
- Los años anteriores a 2018 tienen cobertura parcial: comparar contra ellos daría una falsa sensación de crecimiento.
- La hora está redondeada por el oficial (exceso en :00 y :30): se trabaja a nivel de hora, no de minuto.
- ~95 % de los registros figuran **en intersección**. O la base prioriza la carga de siniestros en cruces, o el campo se completa de forma laxa: no se puede comparar con rigor intersección vs. mitad de cuadra.
- El campo `tipo_choque` ("heridos y/o grúa") **no es** un indicador de heridos: mezcla siniestros con grúa sin lesionados. Para severidad se usa el desglose de lesiones, que es consistente con `lesion_mas_grave`.

## 3. Composición de los siniestros

### 3.1 Severidad: la pirámide de lesiones

In [ ]:
total = len(df)
kpis = pd.Series({
    "Siniestros": total,
    "Con heridos": df["con_heridos"].sum(),
    "Graves (fatal o incapacitante)": df["grave"].sum(),
    "Fatales": df["fatal"].sum(),
    "Personas fallecidas": df["lesiones_fatales"].sum(),
    "Personas con lesión incapacitante": df["lesiones_incapacitantes"].sum(),
})
display(pd.DataFrame({"cantidad": kpis, "% de siniestros": [100, *(kpis.iloc[1:4] / total * 100), np.nan, np.nan]}))

piramide = df["severidad"].value_counts().reindex(NIVELES)
fig, ax = plt.subplots(figsize=(10, 3.4))
colores = [ROJO, NARANJA, AZUL, AZUL_CLARO, "#c3c2b7"]
ax.barh(NIVELES[::-1], piramide.values[::-1], color=colores[::-1], height=0.7)
for i, v in enumerate(piramide.values[::-1]):
    ax.text(v, i, f"  {v:,} ({v / total:.1%})", va="center", fontsize=9, color=TINTA2)
ax.set_xlim(0, piramide.max() * 1.25)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_title("Siniestros según la lesión más grave registrada")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### 3.2 Evolución anual — ¿aumentan los siniestros con heridos?

El brief menciona que "en los últimos dos años los siniestros con heridos vienen en aumento". Se valida con datos.

In [ ]:
anual = df.groupby("anio").agg(
    siniestros=("grave", "size"), con_heridos=("con_heridos", "sum"),
    graves=("grave", "sum"), fatales=("fatal", "sum"))
anual["tasa_heridos_%"] = anual["con_heridos"] / anual["siniestros"] * 100
anual["tasa_graves_%"] = anual["graves"] / anual["siniestros"] * 100
anual["var_heridos_%"] = anual["con_heridos"].pct_change() * 100
display(anual)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, col, color, titulo in zip(axes, ["siniestros", "con_heridos", "graves"], [AZUL, NARANJA, ROJO],
                                  ["Siniestros totales", "Siniestros con heridos", "Siniestros graves"]):
    ax.plot(anual.index, anual[col], color=color, lw=2, marker="o", ms=6)
    for x, v in anual[col].items():
        ax.annotate(f"{v:,}", (x, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7, color=TINTA2)
    ax.set_ylim(0, anual[col].max() * 1.2)
    ax.set_title(titulo)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
plt.tight_layout(); plt.show()

### 3.3 Tipo de choque

In [ ]:
t_choque = panel_volumen_severidad(df, "primer_tipo_choque", "Tipo de choque: volumen vs. severidad", min_n=100)
t_choque[["siniestros", "pct_total", "tasa_graves", "RR_graves", "graves", "fatales"]].round(2)

### 3.4 Causa principal atribuida

In [ ]:
t_causa = panel_volumen_severidad(df, "causa_principal", "Causa principal: volumen vs. severidad (top 20 por volumen)",
                                  min_n=200, top=20)
t_causa[["siniestros", "pct_total", "tasa_graves", "RR_graves", "graves", "fatales"]].round(2)

In [ ]:
# Causas poco frecuentes pero muy severas (≥ 100 casos, RR ≥ 1.5)
tabla_tasas(df, "causa_principal", min_n=100).query("RR_graves >= 1.5").sort_values("RR_graves", ascending=False)[
    ["siniestros", "tasa_graves", "ic_inf", "ic_sup", "RR_graves", "graves", "fatales"]].round(2)

### 3.5 Daño material y vehículos involucrados

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.2))
orden_danos = ["$500 O MENOS", "$501 - $1,500", "SOBRE $1,500"]
d = df["danos"].value_counts(normalize=True).reindex(orden_danos) * 100
a1.bar(["≤ $500", "$501–1.500", "> $1.500"], d.values, color=AZUL, width=0.6)
for x, v in enumerate(d.values):
    a1.text(x, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9, color=TINTA2)
a1.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a1.set_title("Magnitud del daño material")
a1.grid(axis="x", visible=False)

veh = df["vehiculos"].clip(upper=4).map({1: "1", 2: "2", 3: "3", 4: "4 o más"})
tv = tabla_tasas(df.assign(veh=veh), "veh", orden=["1", "2", "3", "4 o más"])
a2.bar(tv.index, tv["tasa_graves"], color=AZUL, width=0.6)
a2.errorbar(tv.index, tv["tasa_graves"], yerr=[tv["tasa_graves"] - tv["ic_inf"], tv["ic_sup"] - tv["tasa_graves"]],
            fmt="none", ecolor=TINTA2, capsize=3)
for x, (v, n) in enumerate(zip(tv["tasa_graves"], tv["siniestros"])):
    a2.text(x, v, f"  {v:.1f}%\n  n={n:,}", ha="left", va="bottom", fontsize=8, color=TINTA2)
a2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a2.set_xlabel("Vehículos involucrados")
a2.set_title("% graves según cantidad de vehículos"); a2.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 4. Patrones temporales y condiciones del entorno

### 4.1 Hora del día

In [ ]:
th = tabla_tasas(df, "hora_num", orden=range(24))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.bar(th.index, th["siniestros"], color=AZUL, width=0.75)
a1.set_title("Siniestros por hora del día (volumen = exposición)")
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}")); a1.grid(axis="x", visible=False)

a2.fill_between(th.index, th["ic_inf"], th["ic_sup"], color=ROJO, alpha=0.15, lw=0)
a2.plot(th.index, th["tasa_graves"], color=ROJO, lw=2, marker="o", ms=5)
a2.axhline(df["grave"].mean() * 100, color=TINTA2, ls="--", lw=1)
a2.set_title("% de siniestros graves por hora (banda: IC 95%) — riesgo real")
a2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
a2.set_xticks(range(24)); a2.set_xlabel("Hora")
plt.tight_layout(); plt.show()

### 4.2 Día de la semana × hora

In [ ]:
conteo = df.pivot_table(index="dia", columns="hora_num", values="grave", aggfunc="size", observed=True)
tasa = df.pivot_table(index="dia", columns="hora_num", values="grave", aggfunc="mean", observed=True) * 100

fig, (a1, a2) = plt.subplots(2, 1, figsize=(15, 7.5))
heatmap(conteo, "Siniestros por día y hora (volumen)", CMAP_AZUL, ax=a1, cbar_label="siniestros")
heatmap(tasa, "% de siniestros graves por día y hora (riesgo)", CMAP_ROJO, fmt=".1f", ax=a2, cbar_label="% graves")
a2.set_xlabel("Hora")
plt.tight_layout(); plt.show()

In [ ]:
tabla_tasas(df, "dia", orden=DIAS)[["siniestros", "pct_total", "tasa_heridos", "tasa_graves", "ic_inf", "ic_sup", "RR_graves"]]

### 4.3 Mes del año

In [ ]:
tm = tabla_tasas(df, "mes_nombre", orden=MESES)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.4))
a1.bar(tm.index, tm["siniestros"] / (ANIO_HASTA - ANIO_DESDE + 1), color=AZUL, width=0.7)
a1.set_title("Promedio de siniestros por mes"); a1.grid(axis="x", visible=False)
a1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
a2.plot(tm.index, tm["tasa_graves"], color=ROJO, lw=2, marker="o")
a2.fill_between(range(12), tm["ic_inf"], tm["ic_sup"], color=ROJO, alpha=0.15, lw=0)
a2.axhline(df["grave"].mean() * 100, color=TINTA2, ls="--", lw=1)
a2.set_title("% de siniestros graves por mes"); a2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
plt.tight_layout(); plt.show()

# Serie mensual completa (dos paneles separados, no doble eje)
serie = df.set_index("fecha").resample("MS").agg({"grave": ["size", "sum"], "con_heridos": "sum"})
serie.columns = ["siniestros", "graves", "con_heridos"]
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(serie.index, serie["siniestros"], color=AZUL, lw=1.5, label="Siniestros")
axes[0].plot(serie.index, serie["con_heridos"], color=NARANJA, lw=1.5, label="Con heridos")
axes[0].legend(frameon=False, loc="upper left"); axes[0].set_title("Serie mensual de siniestros")
axes[1].plot(serie.index, serie["graves"], color=ROJO, lw=1.5)
axes[1].plot(serie.index, serie["graves"].rolling(12, center=True).mean(), color=TINTA2, lw=1, ls="--", label="media móvil 12 meses")
axes[1].legend(frameon=False, loc="upper left"); axes[1].set_title("Serie mensual de siniestros graves")
plt.tight_layout(); plt.show()

### 4.4 Clima e iluminación

In [ ]:
t_luz = panel_volumen_severidad(df, "luz_g", "Condición de iluminación", orden=ORDEN_LUZ, min_n=100)
t_clima = panel_volumen_severidad(df, "clima_g", "Condición climática", orden=ORDEN_CLIMA, min_n=100)
pd.concat({"Iluminación": t_luz, "Clima": t_clima})[["siniestros", "pct_total", "tasa_heridos", "tasa_graves", "RR_graves"]]

In [ ]:
t_sup = panel_volumen_severidad(df, "superficie", "Estado de la superficie de la calzada", min_n=100)

### 4.5 Cruces: hora × iluminación y clima × iluminación

Se muestran solo las celdas con al menos 200 siniestros, para no sobreinterpretar tasas calculadas con pocos casos.

In [ ]:
MIN_CELDA = 200
luz_validas = ORDEN_LUZ[:-1]
sub = df[df["luz_g"].isin(luz_validas)]

def tasa_cruce(data, fila, col, orden_f=None, orden_c=None):
    g = data.groupby([fila, col], observed=True)["grave"].agg(["mean", "size"]).reset_index()
    t = g.pivot(index=fila, columns=col, values="mean") * 100
    n = g.pivot(index=fila, columns=col, values="size")
    if orden_f is not None: t, n = t.reindex(orden_f), n.reindex(orden_f)
    if orden_c is not None: t, n = t[orden_c], n[orden_c]
    return t, n

t1, n1 = tasa_cruce(sub, "luz_g", "franja", luz_validas, FRANJAS)
t2, n2 = tasa_cruce(sub[sub["clima_g"] != "Desconocido"], "clima_g", "luz_g", ORDEN_CLIMA[:-1], luz_validas)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.2))
heatmap(t1, "% graves: iluminación × franja horaria", CMAP_ROJO, ".1f", a1, "% graves", mask=n1.fillna(0) < MIN_CELDA)
heatmap(t2, "% graves: clima × iluminación", CMAP_ROJO, ".1f", a2, "% graves", mask=n2.fillna(0) < MIN_CELDA)
plt.tight_layout(); plt.show()
print("Cantidad de siniestros por celda (clima × iluminación):")
n2

#### Hipótesis del cliente: "más siniestros graves de noche con lluvia"

Se compara la tasa de graves en **noche + lluvia** contra el escenario de referencia **día + despejado**, con su riesgo relativo e intervalo de confianza, y un test chi-cuadrado.

In [ ]:
def comparar(mask_a, mask_b, etiqueta_a, etiqueta_b, data=df):
    a, b = data.loc[mask_a, "grave"], data.loc[mask_b, "grave"]
    ka, na, kb, nb = a.sum(), len(a), b.sum(), len(b)
    rr = (ka / na) / (kb / nb)
    se = np.sqrt(1 / ka - 1 / na + 1 / kb - 1 / nb)
    lo, hi = np.exp(np.log(rr) - 1.96 * se), np.exp(np.log(rr) + 1.96 * se)
    chi2, p, _, _ = stats.chi2_contingency([[ka, na - ka], [kb, nb - kb]])
    return pd.Series({"grupo": etiqueta_a, "referencia": etiqueta_b, "n_grupo": na, "tasa_grupo_%": ka / na * 100,
                      "n_ref": nb, "tasa_ref_%": kb / nb * 100, "RR": rr, "RR_IC_inf": lo, "RR_IC_sup": hi, "p_valor": p})

noche = df["luz_g"].isin(["Noche con alumbrado", "Noche sin alumbrado"])
dia_desp = (df["luz_g"] == "Día") & (df["clima_g"] == "Despejado")
lluvia = df["clima_g"] == "Lluvia"
despejado = df["clima_g"] == "Despejado"
hipotesis = pd.DataFrame([
    comparar(noche & lluvia, dia_desp, "Noche + lluvia", "Día + despejado"),
    comparar(noche & despejado, dia_desp, "Noche + despejado", "Día + despejado"),
    comparar(lluvia & (df["luz_g"] == "Día"), dia_desp, "Día + lluvia", "Día + despejado"),
    comparar(df["luz_g"] == "Noche sin alumbrado", df["luz_g"] == "Noche con alumbrado", "Noche sin alumbrado", "Noche con alumbrado"),
])
hipotesis.round(3)

### 4.6 ¿Qué variables se asocian más con la severidad?

Test chi-cuadrado de independencia entre cada variable y el indicador *grave*, con la **V de Cramér** como medida de fuerza. Con más de 180.000 registros casi todo da "significativo" (p < 0,05), así que lo relevante para priorizar es la **V de Cramér**, no el p-valor.

In [ ]:
variables = ["primer_tipo_choque", "causa_principal", "franja", "hora_num", "dia", "mes", "luz_g", "clima_g",
             "superficie", "tipo_via", "dispositivo_control", "en_interseccion", "alineacion", "defecto_via", "danos"]
asoc = pd.DataFrame([(v, *cramers_v(df[v], df["grave"])) for v in variables],
                    columns=["variable", "V_Cramer", "chi2", "p_valor"]).sort_values("V_Cramer", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(asoc["variable"][::-1], asoc["V_Cramer"][::-1], color=AZUL, height=0.65)
for i, v in enumerate(asoc["V_Cramer"][::-1]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=8, color=TINTA2)
ax.set_title("Fuerza de asociación con 'siniestro grave' (V de Cramér)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
asoc.round(4)

## 5. Puntos críticos geográficos

Se usan los siniestros con coordenadas recuperadas (sección 2.6).

### 5.1 Distribución espacial

Cada hexágono agrupa los siniestros de una zona de ~500 m. Izquierda: todos los siniestros (exposición). Derecha: solo los graves.

In [ ]:
geo = df[df["latitud"].notna()].copy()
ASPECTO = 1 / np.cos(np.radians(geo["latitud"].mean()))  # corrige la deformación lat/lon

fig, axes = plt.subplots(1, 2, figsize=(13, 7.5))
for ax, datos, cmap, titulo in [
    (axes[0], geo, CMAP_AZUL, f"Todos los siniestros (n = {len(geo):,})"),
    (axes[1], geo[geo["grave"]], CMAP_ROJO, f"Siniestros graves (n = {geo['grave'].sum():,})"),
]:
    hb = ax.hexbin(datos["longitud"], datos["latitud"], gridsize=55, cmap=cmap, mincnt=1, linewidths=0.2,
                   extent=(geo["longitud"].min(), geo["longitud"].max(), geo["latitud"].min(), geo["latitud"].max()))
    fig.colorbar(hb, ax=ax, shrink=0.5, label="siniestros por hexágono")
    ax.set_aspect(ASPECTO)
    ax.set_title(titulo)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
fig.suptitle(f"Distribución geográfica de los siniestros — Chicago {ANIO_DESDE}-{ANIO_HASTA}", x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### 5.2 Puntos críticos: grilla de ~150 m e índice de severidad

Para identificar **lugares concretos** (cruces o tramos cortos), la ciudad se divide en una grilla de celdas de ~150 m. En cada celda se calcula:

- **siniestros**, **con heridos**, **graves** y **fatales**;
- un **índice de severidad ponderado** (tipo *EPDO*, *Equivalent Property Damage Only*), que da más peso a lo más grave: fatal = 10, grave (incapacitante) = 5, otra lesión = 1, solo daños = 0. Así, una esquina con pocos choques pero con muertos puede quedar por encima de una con muchos choques leves, y los choques sin heridos (que no son un problema de *seguridad*) no inflan el ranking;
- el **nombre del lugar**, armado con las dos calles más frecuentes en las actas de esa celda (normalmente, las dos calles del cruce).

In [ ]:
CELDA = 0.0015  # grados: ~165 m norte-sur x ~125 m este-oeste
geo["peso"] = np.select([geo["fatal"], geo["grave"], geo["con_heridos"]], [10, 5, 1], default=0)
geo["celda"] = ((geo["latitud"] / CELDA).round().astype(int).astype(str) + "_" +
                (geo["longitud"] / CELDA).round().astype(int).astype(str))
geo["via"] = (geo["direccion_calle"].fillna("") + " " + geo["calle"].fillna("")).str.strip().str.title().str.replace(r"(\d)(St|Nd|Rd|Th)\b", lambda m: m[1] + m[2].lower(), regex=True)
geo["nocturno"] = geo["luz_g"].isin(["Noche con alumbrado", "Noche sin alumbrado"])


def nombre_lugar(vias):
    top = vias[vias != ""].value_counts().index[:2]
    return " / ".join(top)


def moda(s):
    return s.mode().iat[0] if not s.mode().empty else None


celdas = geo.groupby("celda").agg(
    lugar=("via", nombre_lugar), siniestros=("grave", "size"), con_heridos=("con_heridos", "sum"),
    graves=("grave", "sum"), fatales=("fatal", "sum"), indice=("peso", "sum"),
    lat=("latitud", "mean"), lon=("longitud", "mean"),
    pct_nocturno=("nocturno", "mean"), tipo_choque_principal=("primer_tipo_choque", moda),
).sort_values("indice", ascending=False)
celdas["tasa_graves"] = celdas["graves"] / celdas["siniestros"] * 100
celdas["pct_nocturno"] *= 100
celdas.insert(0, "ranking", range(1, len(celdas) + 1))

print(f"Celdas con al menos un siniestro: {len(celdas):,}")
TOP_N = 15
top_puntos = celdas.head(TOP_N)
top_puntos[["ranking", "lugar", "siniestros", "con_heridos", "graves", "fatales", "tasa_graves", "indice",
            "pct_nocturno", "tipo_choque_principal"]].round(1).reset_index(drop=True)

In [ ]:
# ¿Qué tan concentrado está el riesgo? Curva de concentración de siniestros graves por celda
orden = celdas.sort_values("graves", ascending=False)
acum = orden["graves"].cumsum() / orden["graves"].sum() * 100
pct_celdas = np.arange(1, len(orden) + 1) / len(orden) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pct_celdas, acum.values, color=ROJO, lw=2)
ax.plot([0, 100], [0, 100], color=GRIS, ls="--", lw=1)
for p in [1, 5, 10]:
    v = acum.values[int(len(orden) * p / 100) - 1]
    ax.scatter(p, v, color=ROJO, s=40, zorder=3)
    ax.annotate(f"{p}% de las celdas → {v:.0f}% de los graves", (p, v), textcoords="offset points",
                xytext=(10, -12), fontsize=8, color=TINTA2)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("% de celdas con siniestros (ordenadas de más a menos graves)")
ax.set_ylabel("% acumulado de siniestros graves")
ax.set_title("Concentración geográfica de los siniestros graves")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.42 * TOP_N + 1.2))
t = top_puntos.iloc[::-1]
etiquetas = [f"#{r} {l}" for r, l in zip(t["ranking"], t["lugar"])]
otros = t["indice"] - t["fatales"] * 10 - (t["graves"] - t["fatales"]) * 5  # heridos leves
ax.barh(etiquetas, t["fatales"] * 10, color="#8f1d1c", height=0.7, label="Aporte de fatales")
ax.barh(etiquetas, (t["graves"] - t["fatales"]) * 5, left=t["fatales"] * 10, color=ROJO, height=0.7,
        label="Aporte de graves no fatales", edgecolor=FONDO)
ax.barh(etiquetas, otros, left=t["fatales"] * 10 + (t["graves"] - t["fatales"]) * 5, color=AZUL_CLARO, height=0.7,
        label="Aporte de heridos leves", edgecolor=FONDO)
for i, (v, n) in enumerate(zip(t["indice"], t["siniestros"])):
    ax.text(v, i, f"  {v:.0f}  ({n} siniestros)", va="center", fontsize=8, color=TINTA2)
ax.set_xlim(0, t["indice"].max() * 1.25)
ax.set_title(f"Top {TOP_N} puntos críticos según índice de severidad ponderado")
ax.legend(frameon=False, loc="lower right", fontsize=8)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### 5.3 Mapa interactivo de puntos críticos

Mapa de calor de los **siniestros graves**, con los puntos del top marcados en rojo (al pasar el mouse se ve el detalle). También se guarda como `mapa_hotspots.html`, que se abre en cualquier navegador sin necesidad de Python.

In [ ]:
import folium
from folium.plugins import HeatMap

mapa = folium.Map(location=[geo["latitud"].median(), geo["longitud"].median()], zoom_start=11, tiles="cartodbpositron")
HeatMap(geo.loc[geo["grave"], ["latitud", "longitud"]].values.tolist(), radius=9, blur=12,
        name="Siniestros graves (mapa de calor)").add_to(mapa)
capa_top = folium.FeatureGroup(name=f"Top {TOP_N} puntos críticos").add_to(mapa)
for _, r in top_puntos.iterrows():
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=7 + r["graves"] ** 0.5, color=ROJO, fill=True, fill_opacity=0.8, weight=2,
        tooltip=(f"<b>#{r['ranking']} {r['lugar']}</b><br>{r['siniestros']} siniestros · {r['graves']} graves · "
                 f"{r['fatales']} fatales<br>Índice: {r['indice']:.0f} · Nocturnos: {r['pct_nocturno']:.0f}%"),
    ).add_to(capa_top)
folium.LayerControl().add_to(mapa)
mapa.save("mapa_hotspots.html")
mapa

### 5.4 Riesgo por contexto vial

Además de *dónde*, interesa *qué tipo de lugar* es más riesgoso, para extrapolar las intervenciones a sitios con las mismas características.

In [ ]:
t_via = panel_volumen_severidad(df, "tipo_via", "Tipo de vía / intersección", min_n=150)
t_disp = panel_volumen_severidad(df, "dispositivo_control", "Dispositivo de control de tránsito", min_n=150)
t_alin = panel_volumen_severidad(df, "alineacion", "Alineación de la calzada", min_n=150)
t_def = panel_volumen_severidad(df, "defecto_via", "Defecto de la vía", min_n=100)

In [ ]:
# Matriz de riesgo: tipo de vía × dispositivo de control
top_via = df["tipo_via"].value_counts().index[:8]
top_disp = ["SEMÁFORO", "SEÑAL DE PARE/LUZ INTERMITENTE", "SIN CONTROLES"]
sub = df[df["tipo_via"].isin(top_via) & df["dispositivo_control"].isin(top_disp)]
t3, n3 = tasa_cruce(sub, "tipo_via", "dispositivo_control", list(top_via), top_disp)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.6))
heatmap(n3, "Siniestros: tipo de vía × control", CMAP_AZUL, ",.0f", a1, "siniestros")
heatmap(t3, "% graves: tipo de vía × control", CMAP_ROJO, ".1f", a2, "% graves", mask=n3.fillna(0) < MIN_CELDA)
a2.set_yticklabels([])
plt.tight_layout(); plt.show()

## 6. Segmentación por causa principal → palanca de inversión

Las 40 causas del acta policial se agrupan en **familias de conducta o entorno**, y cada familia se asigna a la **palanca de intervención** que mejor la ataca:

| Palanca | Familias de causa | Racional |
|---|---|---|
| **Fiscalización** | Velocidad, alcohol/drogas, desobediencia de señales | Conductas que se corrigen con controles, radares y sanciones |
| **Concientización** | Maniobras imprudentes, distracción, habilidad/condición del conductor | Conductas de riesgo sin "infracción visible" fácil de controlar; responden a educación vial |
| **Infraestructura** | Entorno y vía (visión obstruida, defectos, obras, clima, paradas de colectivo) | Se mitigan con obra: iluminación, señalización, poda, reductores, mantenimiento |
| **Sin determinar** | No se pudo determinar / no aplica | No orientan inversión — son un problema de calidad del dato |

> El agrupamiento es una propuesta del equipo y debe **validarse con el Observatorio Vial** (algunas causas, como *no ceder el paso*, podrían atacarse tanto con fiscalización como con rediseño de la intersección).

In [ ]:
t_fam = tabla_tasas(df, "familia_causa")
t_fam["pct_de_graves"] = t_fam["graves"] / df["grave"].sum() * 100
t_fam["palanca"] = t_fam.index.map(PALANCA)
t_fam[["palanca", "siniestros", "pct_total", "tasa_heridos", "tasa_graves", "RR_graves", "graves", "pct_de_graves", "fatales"]].round(2)


In [ ]:
COLOR_PALANCA = {"Fiscalización": AZUL, "Concientización": NARANJA, "Infraestructura": VERDE, "Otros": GRIS, "Sin determinar": "#c3c2b7"}
fig, ax = plt.subplots(figsize=(11, 6))
DESPLAZAMIENTO = {"Sin determinar": (-80, 24), "Maniobras imprudentes": (-50, -40)}
for fam, r in t_fam.iterrows():
    ax.scatter(r["pct_total"], r["tasa_graves"], s=r["graves"] * 1.2 + 30, color=COLOR_PALANCA[r["palanca"]],
               alpha=0.85, edgecolor=FONDO, linewidth=2, zorder=3)
    ax.annotate(f"{fam}\n{r['graves']:,} graves", (r["pct_total"], r["tasa_graves"]), textcoords="offset points",
                xytext=DESPLAZAMIENTO.get(fam, (10, 4)), fontsize=8, color=TINTA)
ax.axhline(df["grave"].mean() * 100, color=TINTA2, ls="--", lw=1)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlabel("% del total de siniestros (escala log)"); ax.set_ylabel("% de siniestros graves")
ax.set_title("Familias de causa: frecuencia vs. severidad (tamaño = cantidad de graves)")
for pal, c in COLOR_PALANCA.items():
    ax.scatter([], [], color=c, s=60, label=pal)
ax.legend(frameon=False, title="Palanca", loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
# Aporte de cada palanca: sobre el total y sobre los siniestros con causa determinada
ORDEN_PAL = ["Fiscalización", "Concientización", "Infraestructura", "Otros", "Sin determinar"]
t_pal = tabla_tasas(df, "palanca", orden=ORDEN_PAL)
det = df[df["palanca"] != "Sin determinar"]
t_pal["pct_graves_total"] = t_pal["graves"] / df["grave"].sum() * 100
t_pal["pct_graves_con_causa"] = (t_pal["graves"] / det["grave"].sum() * 100).where(t_pal.index != "Sin determinar")

fig, ax = plt.subplots(figsize=(11, 2.8))
izq = 0
for pal, v in t_pal["pct_graves_total"].items():
    ax.barh(0, v, left=izq, color=COLOR_PALANCA[pal], edgecolor=FONDO, linewidth=2, height=0.6)
    if v > 4:
        ax.text(izq + v / 2, 0, f"{pal}\n{v:.0f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    else:
        ax.annotate(f"{pal} {v:.1f}%", (izq + v / 2, -0.3), xytext=(0, -14 if pal == "Otros" else -2),
                    textcoords="offset points", ha="center", va="top", fontsize=8, color=TINTA2)
    izq += v
ax.set_xlim(0, 100); ax.set_ylim(-0.8, 0.4); ax.set_yticks([]); ax.grid(False)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("¿Qué palanca podría atacar cada siniestro grave? (% del total de graves)")
plt.tight_layout(); plt.show()
t_pal[["siniestros", "pct_total", "graves", "tasa_graves", "RR_graves", "pct_graves_total", "pct_graves_con_causa", "fatales"]].round(2)

### 6.1 ¿Cuándo ocurre cada familia de causa? (insumo para patrullaje)

Distribución de los **siniestros graves** de cada familia por franja horaria y fin de semana. Sirve para decidir *cuándo* reforzar los controles de velocidad y alcoholemia.

In [ ]:
graves = df[df["grave"] & (df["palanca"] != "Sin determinar")]
dist = pd.crosstab(graves["familia_causa"], graves["franja"], normalize="index") * 100
dist["% en fin de semana"] = graves.groupby("familia_causa")["fin_de_semana"].mean() * 100
dist["graves (n)"] = graves["familia_causa"].value_counts()
dist = dist.sort_values("graves (n)", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.2))
heatmap(dist[FRANJAS], "% de los siniestros graves de cada familia según franja horaria", CMAP_AZUL, ".0f", ax, "% de la fila")
plt.tight_layout(); plt.show()
dist.round(1)

In [ ]:
# Usuarios vulnerables: peatones y ciclistas
vuln = df["primer_tipo_choque"].isin(["PEATÓN", "CICLISTA"])
resumen_vuln = pd.DataFrame({
    "siniestros": [vuln.sum(), (~vuln).sum()],
    "% del total": [vuln.mean() * 100, (~vuln).mean() * 100],
    "graves": [df.loc[vuln, "grave"].sum(), df.loc[~vuln, "grave"].sum()],
    "fatales": [df.loc[vuln, "fatal"].sum(), df.loc[~vuln, "fatal"].sum()],
}, index=["Peatón o ciclista", "Resto"])
resumen_vuln["% de graves"] = resumen_vuln["graves"] / df["grave"].sum() * 100
resumen_vuln["% de fatales"] = resumen_vuln["fatales"] / df["fatal"].sum() * 100
resumen_vuln.round(1)

## 7. Hallazgos principales

Período analizado: **2018–2024**, 181.051 siniestros. Cifras redondeadas; el detalle está en las tablas de cada sección.

**1. Composición: la mayoría son choques solo con daños, pero la fracción grave no es despreciable.**
El 72,9 % de los siniestros no tiene lesionados. El 27,1 % tiene al menos un herido y el **3,4 % es grave** (6.146 hechos con muertos o heridos incapacitantes). Hubo 321 siniestros fatales, con 355 personas fallecidas.

**2. La percepción del cliente se confirma en parte.**
Los siniestros **con heridos** llegaron en 2024 a su máximo del período (7.731, un +9,8 % respecto de 2023), y la proporción de siniestros con heridos subió del 25,1 % (2019) al 29,5 % (2024). En cambio, los siniestros **graves** no aumentaron: 2024 es el año con menos graves (767). Ese último dato hay que tomarlo con cautela, porque diciembre de 2024 tiene muchos menos graves que otros diciembres con un volumen total similar. Puede ser que las lesiones todavía no se hayan reclasificado (ver limitaciones).

**3. Peatones y ciclistas son el foco número uno.**
Representan el **7 % de los siniestros, pero el 32 % de los graves y el 32 % de los fatales**. En un atropello a peatón, el 18,4 % de los hechos son graves, más de 5 veces el promedio. El tipo de choque es la variable más asociada a la severidad (V de Cramér 0,20).

**4. El volumen y el riesgo ocurren en horarios distintos.**
La mayor cantidad de siniestros se da entre las 15 y las 17 h (hora pico). La mayor **probabilidad de que un siniestro sea grave** se da entre las **0 y las 6 h** (pico del 5,9 % a las 2 h) y vuelve a subir desde las 20 h. La madrugada concentra el 9 % de los siniestros pero el 13,8 % de los graves. El **domingo** es el día más riesgoso (RR 1,21) y le sigue el sábado.

**5. "De noche con lluvia": lo que pesa es la noche, no la lluvia.**
Comparado con *día + despejado* (3,2 % de graves), un siniestro de *noche + lluvia* tiene 1,28 veces más chances de ser grave (IC 1,14–1,43). Pero de *noche con cielo despejado* el riesgo es todavía mayor (RR 1,42), y *día + lluvia* no se distingue del día despejado (RR 0,92, p = 0,16). La lluvia no aumenta la gravedad y la nieve incluso la reduce (2,2 % de graves, RR 0,65), probablemente porque se circula más despacio. Sin datos de tránsito no se puede saber si aumentan la *cantidad* de choques. Tampoco aparece diferencia de severidad entre noche con y sin alumbrado (RR 0,99). Antes de concluir que la iluminación "no sirve", hay que tener en cuenta que el campo puede estar mal cargado y que las calles sin alumbrado pueden tener otro perfil de tránsito.

**6. Causas: hay pocas causas de mucho volumen y otras de baja frecuencia pero letales.**
- **No ceder el derecho de paso** es la causa con más graves en números absolutos (1.687) por su volumen.
- **No respetar los semáforos** es la causa determinada con **más fatales** (67), con una tasa de graves del 6,5 %, casi el doble del promedio.
- Las causas de **alcohol/drogas** (≈10 % de graves, RR 2,9) y **exceso de velocidad** (12,8 %, RR 3,8) son poco frecuentes pero muy severas. Los graves con alcohol ocurren en un 39 % de madrugada y en un 51 % en fin de semana.

**7. Palancas de inversión.**
Tomando los graves con causa determinada, el **57 %** corresponde a conductas que se atacan con **concientización** (maniobras imprudentes, distracción, impericia), el **38 %** con **fiscalización** (velocidad, alcohol, desobediencia de señales) y solo el **4 %** con **infraestructura**. Pero la fiscalización actúa sobre los siniestros **más severos por hecho** (5,4 % de graves, RR 1,6): cada siniestro que evita "rinde" más. El peso bajo de infraestructura está casi seguro subestimado, porque el acta policial tiende a atribuir la causa al conductor y no a la vía.

**8. El contexto vial explica poco por sí solo.**
El tipo de vía, el dispositivo de control, la alineación y los defectos tienen una asociación débil con la severidad (V ≤ 0,03). Lo que más pesa es **quién está involucrado** (peatón, ciclista) y **qué conducta** hubo. Para priorizar, entonces, no sirve "intervenir todas las avenidas de cierto tipo": hay que ir a **lugares concretos**.

**9. El riesgo grave está muy concentrado geográficamente.**
Las coordenadas se recuperaron para el 99,4 % de los siniestros. De las ~15.400 celdas de ~150 m con algún siniestro, **el 1 % concentra el 16 % de los siniestros graves y el 10 % concentra dos tercios (67 %)**. Las zonas de mayor densidad son el centro (Loop / Near West Side) y corredores del West Side y del South Side. Intervenir en pocas decenas de puntos cubre una parte relevante del problema.
- **E Lower Wacker Dr / N Columbus Dr** encabeza el ranking por índice de severidad (19 graves, 1 fatal).
- **N Pulaski Rd / W Madison St** es el caso más llamativo: 18 graves en solo 145 siniestros (**12,4 % de graves, 3,6 veces el promedio**).
- **W Harrison St / S Independence Blvd** acumula 3 fatales.
- En varios puntos del top (S Wells St / W Garfield Blvd, S State St / W 71st St) más del 40 % de los siniestros son nocturnos.

### Insumos preliminares para las intervenciones (a validar en Fases 4 y 6)

| Palanca | Indicio de los datos | Intervención candidata |
|---|---|---|
| Fiscalización | Riesgo máximo 0–6 h y fin de semana; alcohol concentrado en esas franjas | Operativos de alcoholemia y velocidad viernes, sábado y domingo de 22 a 6 h |
| Fiscalización / Infraestructura | "No respetar semáforos" = causa determinada con más fatales | Cámaras de foto-rojo y revisión de los tiempos semafóricos en los cruces del top de puntos críticos |
| Infraestructura | 10 % de las celdas concentran el 67 % de los graves; casos como N Pulaski Rd / W Madison St con 3,6 veces el promedio | Auditoría de seguridad vial en los 15 puntos críticos de la sección 5.2, empezando por los de mayor tasa de graves |
| Infraestructura / Concientización | Peatones y ciclistas: 32 % de los graves | Cruces peatonales seguros, reductores de velocidad, campañas dirigidas a conductores en el giro |

## 8. Limitaciones del dato y próximos pasos

### Limitaciones que afectan los hallazgos

| # | Limitación | Impacto | Mitigación aplicada / propuesta |
|---|---|---|---|
| 1 | **La base entregada no trae latitud/longitud ni dirección** | Sin ellas no habría mapa ni ranking de puntos críticos | Se recuperaron del portal de datos abiertos (99,4 % de los registros). Un 4,5 % se vinculó por desempate, con una pequeña probabilidad de asignar la ubicación de un siniestro "gemelo". Pedir que la próxima exportación incluya el ID del siniestro y las coordenadas |
| 1b | **Puntos críticos por celdas de ~150 m** | Una celda puede abarcar dos cruces cercanos; el nombre del lugar se infiere de las calles de las actas | Antes de intervenir, verificar cada punto en terreno o en imagen satelital |
| 2 | **30,6 % de los siniestros sin causa determinada** | La segmentación por causa describe solo a ~70 % de los hechos | Se reportan los porcentajes sobre el total y sobre los siniestros con causa determinada |
| 3 | **Causa subjetiva** (criterio del oficial) y **sin causa secundaria** | Posible sesgo hacia causas "de conductor" en lugar de "de vía" | Se recomienda validar el agrupamiento por palanca con el Observatorio Vial |
| 4 | **Cobertura incompleta antes de 2018** | Comparar con años anteriores daría una falsa tendencia creciente | El análisis usa solo 2018–2024 |
| 5 | **Posible subregistro de graves recientes** (dic-2024) | La baja de graves en 2024 podría ser artificial | Volver a medir cuando la base se actualice; no presentar "baja de graves en 2024" como logro |
| 6 | **Hora redondeada** (36,6 % en minutos múltiplos de 15) | No hay precisión por debajo de la hora | El análisis se hace por hora y por franja |
| 7 | **95 % de los registros figuran en intersección** | No se puede comparar con rigor intersección vs. mitad de cuadra | Se informa como hallazgo de calidad, no como patrón de riesgo |
| 8 | **Faltan campos del brief:** fuga, carriles, fecha de notificación | No se puede analizar la demora en la notificación ni los siniestros con fuga | Incluirlos en la próxima exportación |
| 9 | **Sin datos de exposición** (flujo vehicular por zona y hora) | El volumen alto en hora pico refleja más tránsito, no necesariamente más riesgo | Se usa la tasa de graves *condicional al siniestro* como medida de riesgo |
| 10 | **Campos con "desconocido"** (defecto de vía 16,8 %, superficie 6 %) | Los hallazgos sobre infraestructura son menos robustos | Se excluyen las categorías desconocidas de las comparaciones |

### Próximos pasos (Fase 4: modelo de riesgo relativo)

1. Con la georreferenciación ya es viable un **modelo de riesgo relativo por zona y horario**. Por ejemplo, una regresión logística sobre *grave* con zona, franja horaria, día, iluminación, clima, tipo de choque y familia de causa, interpretable en términos de *odds ratios*. También se puede complementar con un análisis estadístico de clusters espaciales (por ejemplo, Getis-Ord Gi*).
2. Validar con el Observatorio Vial el agrupamiento de causas → palancas, el criterio de severidad (KSI) y los pesos del índice de puntos críticos.
3. Definir la **línea base** de siniestros con heridos y graves por zona (celdas o barrios), para medir el impacto de las intervenciones a 12 meses.

## Anexo: conceptos estadísticos utilizados

### Intervalo de confianza de Wilson

**Qué es:** cuando se calcula un porcentaje a partir de una muestra, ese número tiene un margen de error. El intervalo de Wilson da un **rango donde probablemente está el valor real**, con un 95 % de confianza. Se eligió porque funciona bien incluso con **pocos casos o porcentajes muy chicos**, justo donde la fórmula clásica (p ± 1,96·error estándar) falla. Este proyecto tiene ambos problemas: los siniestros graves son el 3,4 % y varias categorías tienen pocos registros.

**Para qué se usa en este análisis:** para no confundir un porcentaje alto con ruido.

- **Peatón:** 8.024 siniestros y 18,4 % de graves. El intervalo es angosto (≈ 17,6 %–19,3 %), así que el resultado es confiable.
- **Exceder el límite de velocidad:** solo 242 siniestros y 12,8 % de graves. El intervalo es ancho (9,2 %–17,6 %): el riesgo es alto, pero no se conoce con precisión cuánto.

En los gráficos de dos paneles es la **barra horizontal** alrededor de cada punto, y el color sale de comparar el intervalo con el promedio general (3,4 %):

- **Rojo:** el intervalo entero queda por encima del promedio, así que la categoría es más riesgosa que el promedio.
- **Azul:** el intervalo entero queda por debajo, así que es menos riesgosa.
- **Gris:** el intervalo cruza el promedio, así que no se puede afirmar que sea distinta.

### V de Cramér

**Qué es:** un número entre **0 y 1** que mide **qué tan fuerte es la relación entre dos variables categóricas**. Con 0 no hay relación, y con 1 conocer una variable permite saber exactamente la otra. Se calcula a partir del test chi-cuadrado de independencia, ajustado por el tamaño de la muestra y la cantidad de categorías.

**Por qué no alcanza con el chi-cuadrado:** con más de 180.000 registros, el test da "significativo" (p < 0,05) para casi cualquier variable, aunque la relación sea mínima. El p-valor solo responde *si* hay relación, no *qué tan fuerte* es. La V de Cramér responde lo segundo.

**Para qué se usa en este análisis:** para ordenar qué variables se relacionan más con que un siniestro sea grave (sección 4.6).

- **Tipo de choque:** 0,20. Es la variable más fuerte, sobre todo por peatones y ciclistas.
- **Causa principal:** 0,10.
- **Hora, iluminación, clima, tipo de vía:** entre 0,02 y 0,04. La relación es real, pero débil.

Así se fundamenta el hallazgo de que pesa más **quién está involucrado y qué conducta hubo** que las condiciones de la vía o del clima. Como referencia práctica, en tablas grandes una V de ~0,1 indica una relación chica pero relevante, y una de 0,3 o más, una relación fuerte.